In [1]:
import ast

import polars as pl

In [2]:
df = pl.read_csv("../data.csv", separator=";")
df.shape

(1071, 10)

In [3]:
df.columns

['projektname',
 'kurzzusammenfassung',
 'art',
 'einsatzbereich',
 'status',
 'organisation',
 'webseite_link',
 'quelle',
 'lizenz',
 'lizenz_organisation']

## Listen-Spalten parsen

In [4]:
df = df.with_columns(
    pl.col("art").map_elements(ast.literal_eval, return_dtype=pl.List(pl.Utf8)),
    pl.col("einsatzbereich").map_elements(ast.literal_eval, return_dtype=pl.List(pl.Utf8)),
)
df.select("projektname", "art", "einsatzbereich").head()

projektname,art,einsatzbereich
str,list[str],list[str]
"""#AndYou? Politisches Weiterden…","[""Datenanalyse"", ""Digitale Plattformen"", … ""Webanwendungen""]","[""Bildung"", ""Demokratie & Soziale Rechte"", … ""Inklusion & Teilhabe""]"
"""#GenoDigital""","[""Datenerhebung"", ""Open-Source-Software""]","[""Inklusion & Teilhabe"", ""Soziale Dienste""]"
"""#KickHate – Gemeinsam gegen Ha…","[""Automatisierung"", ""Fortbildung"", ""Künstliche Intelligenz""]","[""Anti Dismkriminierung"", ""Inklusion & Teilhabe""]"
"""(Teil)-Automatisierung von Dat…","[""Automatisierung"", ""Datenanalyse"", ""Datenreporting""]","[""Organisation & Professionalisierung"", ""Soziale Dienste""]"
"""1 Bild sagt mehr als 1000 Wort…",[],"[""Bildung"", ""Inklusion & Teilhabe"", ""Soziale Dienste""]"


## Häufigste Kategorien (`art`)


In [5]:
(
    df.select("art")
    .explode("art")
    .group_by("art")
    .len()
    .sort("len", descending=True)
)

/tmp/ipykernel_453312/3849583773.py:3: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("art")


art,len
str,u32
"""Webanwendungen""",474
"""Künstliche Intelligenz""",408
"""Datenreporting""",333
"""Datenanalyse""",320
"""Öffentliche Daten""",210
…,…
"""Bildverarbeitung""",32
"""Wissensorganisation""",27
"""Recomender System""",16


## Häufigste Einsatzbereiche

In [6]:
(
    df.select("einsatzbereich")
    .explode("einsatzbereich")
    .group_by("einsatzbereich")
    .len()
    .sort("len", descending=True)
)

/tmp/ipykernel_453312/399859936.py:3: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("einsatzbereich")


einsatzbereich,len
str,u32
"""Inklusion & Teilhabe""",388
"""Klima & Umwelt""",290
"""Soziale Dienste""",247
"""Stadtentwicklung""",244
"""Bildung""",198
…,…
null,63
"""Internationale Projekte""",59
"""Flucht & Migration""",40


## Verteilung der (normalisierten) Status-Werte

In [7]:
df["status"].value_counts().sort("count", descending=True)

status,count
str,u32
"""In Betrieb""",466
"""Unbekannt""",365
"""Abgeschlossen""",168
"""In Weiterentwicklung""",26
"""In Planung""",23
"""Im Testbetrieb""",14
"""Eingestellt""",8
"""Prototyp""",1


## Beispiel-Filter: alle Projekte einer Kategorie

Zeilen, deren `art`-Liste die Kategorie `Datenanalyse` enthält.

In [8]:
(
    df.filter(pl.col("art").list.contains("Datenanalyse"))
    .select("projektname", "status", "organisation")
    .head()
)

projektname,status,organisation
str,str,str
"""#AndYou? Politisches Weiterden…","""Unbekannt""","""Civic Coding"""
"""(Teil)-Automatisierung von Dat…","""In Betrieb""","""Chancenwerk e.V., CorrelAid e.…"
"""1000 LoRaWAN Nodes im ländlich…","""In Betrieb""","""OK Lab Kreis Schleswig-Flensbu…"
"""112 - KI rettet Leben.""","""In Planung""",null
"""112 - KI rettet Leben""","""In Betrieb""","""Civic Coding"""
